# E-Commerce Customer Segmentation: RFM and Hybrid Clustering
**Prepared by:** Shloke Binani | Academic project

## Notebook Overview
This notebook segments e-commerce customers by purchasing behavior. The data is the public Online Retail transaction dataset (UCI); download `OnlineRetail.csv` and place it next to this notebook to rerun the first cell. The workflow has four phases:
1. **Data loading and SQL (SQLite):** Load the raw transactions into SQLite and calculate Recency, Frequency and Monetary (RFM) values per customer with a SQL query.
2. **Feature scaling:** Standardize the three RFM metrics so each carries equal weight in the distance-based clustering.
3. **Hybrid clustering:** DBSCAN isolates outlier customers, then K-Means groups the remaining customers into three segments.
4. **Validation and profiling:** Check cluster separation with the silhouette score, compare other numbers of clusters, and profile each segment's size and share of revenue.

In [1]:
import sqlite3
import pandas as pd

csv_file = 'OnlineRetail.csv'
db_file = 'ecommerce_data.db'
table_name = 'transactions'
print("Reading CSV file...")
df = pd.read_csv(csv_file, encoding='ISO-8859-1')

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print("Successfully converted 'InvoiceDate' column to a standard format.")

print(f"Connecting to database '{db_file}'...")
conn = sqlite3.connect(db_file)

print(f"Loading cleaned data into SQL table '{table_name}'...")
df.to_sql(table_name, conn, if_exists='replace', index=False)

conn.close()
print("\nSuccess! The database has been created with clean, standardized dates.")

Reading CSV file...
Successfully converted 'InvoiceDate' column to a standard format.
Connecting to database 'ecommerce_data.db'...
Loading cleaned data into SQL table 'transactions'...

Success! The database has been created with clean, standardized dates.


## RFM metrics from SQL

In [2]:
import pandas as pd
import sqlite3

db_file = 'ecommerce_data.db'
conn = sqlite3.connect(db_file)

rfm_query = """
WITH
    -- Find the snapshot date, which is one day after the last transaction in the whole dataset
    snapshot AS (
        SELECT date(MAX(InvoiceDate), '+1 day') AS snapshot_date
        FROM transactions
    ),
    -- Calculate raw RFM metrics for each customer
    rfm_metrics AS (
        SELECT
            CustomerID,
            MAX(date(InvoiceDate)) AS last_purchase_date,
            COUNT(DISTINCT InvoiceNo) AS Frequency,
            SUM(Quantity * UnitPrice) AS MonetaryValue
        FROM
            transactions
        WHERE
            CustomerID IS NOT NULL AND Quantity > 0 AND UnitPrice > 0
        GROUP BY
            CustomerID
    )
-- Final SELECT: Calculate Recency by comparing to the snapshot_date
SELECT
    m.CustomerID,
    (julianday(s.snapshot_date) - julianday(m.last_purchase_date)) AS Recency,
    m.Frequency,
    m.MonetaryValue
FROM
    rfm_metrics m, snapshot s;
"""

print("Executing the final, corrected RFM query...")
rfm_df = pd.read_sql_query(rfm_query, conn)
conn.close()

print("\n--- FINAL CORRECTED RFM DATA ---")
# Now you will see logical, positive Recency values (e.g., 10, 50, 373)
print(rfm_df.head())

Executing the final, corrected RFM query...

--- FINAL CORRECTED RFM DATA ---
   CustomerID  Recency  Frequency  MonetaryValue
0     12346.0    326.0          1       77183.60
1     12347.0      3.0          7        4310.00
2     12348.0     76.0          4        1797.24
3     12349.0     19.0          1        1757.55
4     12350.0    311.0          1         334.40


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import numpy as np

if 'CustomerID' in rfm_df.columns:
    rfm_df.set_index('CustomerID', inplace=True)

# Cluster ONLY on the three RFM metrics (so no leftover columns can leak into the scaling or clustering)
features = ['Recency', 'Frequency', 'MonetaryValue']

print("Part 1: Scaling the RFM data...")
scaler = StandardScaler()
rfm_scaled = pd.DataFrame(scaler.fit_transform(rfm_df[features]), index=rfm_df.index, columns=features)
print("original")
print(rfm_df[features].head())
print("Scaling complete.")
print(rfm_scaled.head())

Part 1: Scaling the RFM data...
original
            Recency  Frequency  MonetaryValue
CustomerID                                   
12346.0       326.0          1       77183.60
12347.0         3.0          7        4310.00
12348.0        76.0          4        1797.24
12349.0        19.0          1        1757.55
12350.0       311.0          1         334.40
Scaling complete.
             Recency  Frequency  MonetaryValue
CustomerID                                    
12346.0     2.329388  -0.425097       8.358668
12347.0    -0.900588   0.354417       0.250966
12348.0    -0.170593  -0.035340      -0.028596
12349.0    -0.740589  -0.425097      -0.033012
12350.0     2.179389  -0.425097      -0.191347


In [4]:

# --- Part 2: Hybrid Clustering ---
print("\nPart 2: Running Hybrid Clustering...")

dbscan = DBSCAN(eps=0.8, min_samples=15)
dbscan.fit(rfm_scaled)
rfm_df['Is_Outlier'] = dbscan.labels_
rfm_df['Is_Outlier'] = rfm_df['Is_Outlier'].apply(lambda x: 1 if x == -1 else 0)

core_customers_df = rfm_df[rfm_df['Is_Outlier'] == 0].copy()
core_customers_scaled = rfm_scaled[rfm_df['Is_Outlier'] == 0].copy()
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(core_customers_scaled)
core_customers_df['Cluster'] = kmeans.labels_

rfm_df['Cluster'] = core_customers_df['Cluster']
rfm_df['Cluster'] = rfm_df['Cluster'].fillna(3) 
rfm_df['Cluster'] = rfm_df['Cluster'].astype(int)
rfm_df.drop(columns=['Is_Outlier'], inplace=True)
print("\nHybrid Clustering Complete. Final Distribution:")
print(rfm_df['Cluster'].value_counts().sort_index())




Part 2: Running Hybrid Clustering...



Hybrid Clustering Complete. Final Distribution:
Cluster
0    2861
1    1053
2     373
3      51
Name: count, dtype: int64


In [5]:
# --- Part 3: Validate the clusters ---
print("\nPart 3: Validating the core clusters...")
score = silhouette_score(core_customers_scaled, kmeans.labels_)
print(f"Silhouette score for k=3 (core customers, excluding the DBSCAN outliers): {score:.3f}")

# Compare other numbers of clusters
print("\nSilhouette score by number of clusters:")
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(core_customers_scaled)
    print(f"  k={k}: {silhouette_score(core_customers_scaled, km.labels_):.3f}")


Part 3: Validating the core clusters...


Silhouette score for k=3 (core customers, excluding the DBSCAN outliers): 0.581

Silhouette score by number of clusters:


  k=2: 0.607


  k=3: 0.581


  k=4: 0.523
  k=5: 0.472


  k=6: 0.432


**Choice of k.** k=2 scores slightly higher (it only separates active from lapsed customers), while k=3 still scores well and splits the active customers into a regular group and a high-frequency, high-spend group that is easier to act on. k=3 was chosen for interpretability, not because it has the top silhouette score.

In [6]:
# --- Part 4: Final profile analysis ---
print("\nPart 4: Calculating final segment profiles...")
segment_profile = rfm_df.groupby('Cluster').agg(
    Recency=('Recency', 'mean'),
    Frequency=('Frequency', 'mean'),
    MonetaryValue=('MonetaryValue', 'mean'),
    CustomerCount=('Recency', 'size'),
    TotalRevenue=('MonetaryValue', 'sum'),
)
segment_profile['Customers_%'] = (segment_profile.CustomerCount / segment_profile.CustomerCount.sum() * 100).round(1)
segment_profile['Revenue_%'] = (segment_profile.TotalRevenue / segment_profile.TotalRevenue.sum() * 100).round(1)
segment_profile = segment_profile.round(2)

print("\n--- FINAL SEGMENT PROFILE ---")
print(segment_profile.sort_values(by='MonetaryValue', ascending=False))


Part 4: Calculating final segment profiles...

--- FINAL SEGMENT PROFILE ---
         Recency  Frequency  MonetaryValue  CustomerCount  TotalRevenue  \
Cluster                                                                   
3          29.86      45.94       56716.04             51    2892518.24   
2          15.36      14.92        5922.97            373    2209267.67   
0          46.57       3.14        1155.78           2861    3306679.78   
1         249.95       1.55         477.63           1053     502942.21   

         Customers_%  Revenue_%  
Cluster                          
3                1.2       32.5  
2                8.6       24.8  
0               66.0       37.1  
1               24.3        5.6  


**Segment labels used in the dashboard.** The labels below describe each segment's RFM pattern. They are interpretations, since the data has no information on who the customers are.

| Cluster | Dashboard label | RFM pattern |
| :--- | :--- | :--- |
| 3 | Anomalous Shopper | 51 outlier customers: very frequent, very high spend (about 1% of customers, about a third of revenue) |
| 2 | High-Value Reseller | 373 customers: very recent, frequent, high spend |
| 0 | Loyal Homemaker | 2,861 customers: fairly recent, a few orders each, moderate spend |
| 1 | At-Risk Shopper | 1,053 customers: inactive for about 250 days on average, low spend |

In [7]:
rfm_df.to_csv('customer_segments_final.csv')
print("Final segmented data has been saved to 'customer_segments_final.csv'")

Final segmented data has been saved to 'customer_segments_final.csv'
